# 50 — Coverage, properties and quality summaries

Regenerate summaries from saved inventory and selected-window metric tables. The current scope is printed first: a pilot is not an archive-wide result. Triggered records cannot establish continuous uptime or an unbiased event/noise population.

Metrics are not recomputed merely to make figures. If review decisions changed, rerun notebook 30. Keep station/channel, gain epoch, clipping, artifacts and record length visible when interpreting any trend.

Fixed/detected windows follow the policy saved by notebook 30; individual manual review is optional.


In [ ]:
from pathlib import Path
import sys
_candidates = [Path.cwd(), Path.cwd() / '20_analysis', *Path.cwd().parents]
ANALYSIS_DIR = next((p for p in _candidates if (p / 'analysis_workflow.py').is_file()), None)
if ANALYSIS_DIR is None:
    raise FileNotFoundError('Start in 20_analysis or its project root, beside analysis_workflow.py.')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))
from analysis_workflow import load_context, describe_context
from IPython.display import display, Image, Markdown
import matplotlib.pyplot as plt
import pandas as pd
ctx = load_context(ANALYSIS_DIR)
display(describe_context(ctx))


In [ ]:
from analysis_workflow import archive_summary, build_inventory, load_metrics, plot_inventory, plot_metric_summary
summary, flag_counts = archive_summary(ctx)
display(pd.DataFrame([summary]))
display(flag_counts)
inventory, files = build_inventory(ctx)
metrics = load_metrics(ctx)
for fig in [plot_inventory(ctx, inventory), plot_metric_summary(ctx, metrics)]:
    plt.close(fig)
for name in ['archive_inventory', 'accepted_window_metrics']:
    display(Image(filename=str(ctx['out'] / 'figures' / f'{name}.png')))


## Per-channel and gain-epoch comparisons

Missing values are not filled with zero. Tables retain each window type separately. A channel/epoch with no selected windows contributes no noise/event estimate. Temporal/shape features on noise are descriptive; threshold-onset features without an external noise reference are intentionally missing.

In [ ]:
if not metrics.empty:
    fields = ['amplitude.rms_counts', 'psd.centroid_hz', 'psd.lf_power_counts2',
              'psd.hf_power_counts2', 'feature.spectral.flatness', 'feature.full.duration_90_s']
    grouped = metrics.groupby(['trace_id', 'gain_epoch_id', 'window'], dropna=False)[fields].agg(['count', 'median'])
    display(grouped)
    grouped.to_csv(ctx['out'] / 'channel_epoch_metric_summary.csv')
else:
    print('No selected-window summaries: complete window review first.')
print('Figure captions are saved beside PNG/PDF/SVG files.')


Use the example spectral-evolution tables produced in notebook **40** to investigate time-frequency transitions; they are not an archive-wide trend until the full sample has been processed. Keep **legacy_rsam_analysis.ipynb** separate: its acquisition clocks, cadence and calibration differ, so join on documented identity/time correspondence rather than assuming alignment.

## EnhancedCatalog / EventRate recording-file counts

After notebook 30, the active sidecars can be loaded without reading waveform samples. EventRate can explicitly use `time_source='recorded_file'`; no earthquake origin is invented. Missing magnitudes remain missing and energy aggregation is disabled. This counts selected recording files per day, **not earthquakes, completeness or continuous network uptime**. Multiple files can represent the same event, and a file can contain several signals.

In [ ]:
from analysis_workflow import load_event_metrics_catalog, save_figure
from flovopy.enhanced.eventrate import EventRateConfig
if (ctx['out'] / 'event_metrics_index.csv').exists():
    catalog = load_event_metrics_catalog(ctx, include_trace_metrics=False)
    rate = catalog.to_event_rate(EventRateConfig(interval='D', time_source='recorded_file', include_energy=False))
    rate.to_csv(str(ctx['out'] / 'recorded_file_rate.csv'))
    display(rate.to_dataframe().head(20))
    if not rate.df.empty:
        fig, ax = plt.subplots(figsize=(7.2, 2.8), layout='constrained')
        rate.plot_event_count(ax=ax, label='Selected waveform files', use_smoothed=False)
        ax.set(ylabel='Selected recording files / day', xlabel='Recorded date (clock uncorrected)')
        save_figure(fig, ctx['out'] / 'figures/recorded_file_rate',
            'Selected waveform-file counts from EnhancedCatalog metadata, aggregated with EventRate. Counts are not earthquake occurrence or station uptime; missing magnitudes are not imputed.',
            ctx['config']['plots']['dpi'])
        display(Image(filename=str(ctx['out'] / 'figures/recorded_file_rate.png')))
        plt.close(fig)
else:
    print('Run notebook 30 first to create the active EnhancedEvent sidecar index.')
